In [ ]:
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
import pandas as pd
from datasets import Dataset
from sklearn.model_selection import train_test_split

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("microsoft/DialoGPT-small")
tokenizer.pad_token = tokenizer.eos_token
tokenizer

In [ ]:
model = AutoModelForCausalLM.from_pretrained("microsoft/DialoGPT-small")
model

In [ ]:
dataset = pd.read_csv("normalized_context_and_response.csv")
dataset.head()

In [ ]:
contexts = dataset['context'].astype('str').values
contexts[:5]

In [ ]:
responses = dataset['response'].astype('str').values
responses[:5]

In [ ]:
def combineText(example):
    return {
        "text": "User: " + example['context'] + 
            "Bot: " + example['response']
    }

In [ ]:
def encode(example):
    return tokenizer(example['text'], truncation=True, padding=True, max_length=64)

In [ ]:
def add_labels(example):
    example['labels']=example['input_ids']
    return example

In [ ]:
datasets = Dataset.from_dict({
    "context": contexts,
    "response": responses
})
datasets

In [ ]:
datasets_split = datasets.train_test_split(test_size=0.2)
datasets_split

In [ ]:
trainSet = datasets_split['train']
trainSet

In [ ]:
testSet = datasets_split['test']
testSet

In [ ]:
trainSet = trainSet.map(combineText)
trainSet

In [ ]:
testSet = testSet.map(combineText)
testSet

In [ ]:
trainSet = trainSet.map(encode, batched=True)
trainSet

In [ ]:
testSet = testSet.map(encode, batched=True)
testSet

In [ ]:
trainSet = trainSet.map(add_labels)
trainSet

In [ ]:
testSet = testSet.map(add_labels)
testSet

In [ ]:
trainingArgs = TrainingArguments(
    output_dir="./output",
    num_train_epochs=2,
    learning_rate=2e-5,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    evaluation_strategy='steps',
    # warmup_steps=500,
    weight_decay=0.01,
    logging_dir=None
)

In [ ]:
trainSet = trainSet.select(range(1100))

In [ ]:
testSet = testSet.select(range(670))

In [ ]:
trainer=Trainer(
    model=model,
    args=trainingArgs,
    train_dataset= trainSet,
    eval_dataset=testSet
)

In [ ]:
trainer.evaluate(testSet)

In [ ]:

trainer.predict(testSet.select(range(80)))

In [ ]:
trainer.train()

In [ ]:
trainer.save_model("mental-health-dialogpt")

In [ ]:
trainer.evaluate(testSet)

In [ ]:
trainer.predict(testSet.select(range(80)))